# Occlusion test on the final fine-tuned SegFormer (epoch 21)
Inputs: `rr-code`, `segformer-ft-model`, DeepGlobe road extraction. Settings: GPU T4 x2, Internet On. Save Version → Save & Run All.

In [1]:
RUN = "occ_segformer_ft"

In [2]:
import glob, os, shutil
hits = glob.glob('/kaggle/input/**/scripts/evaluate_checkpoint.py', recursive=True)
assert hits, "project code dataset is not attached"
PROJECT = os.path.dirname(os.path.dirname(hits[0]))
sat = glob.glob('/kaggle/input/**/*_sat.jpg', recursive=True)
assert sat, "DeepGlobe dataset is not attached - Add Input -> deepglobe road extraction"
masks = [s for s in sat if os.path.exists(s.replace('_sat.jpg', '_mask.png'))]
assert masks, "Found DeepGlobe images but no *_mask.png next to them - wrong DeepGlobe copy"
DEEPGLOBE = os.path.dirname(masks[0])
shutil.copytree(PROJECT, '/kaggle/working/rr', dirs_exist_ok=True)
os.chdir('/kaggle/working/rr')
print("PROJECT  :", PROJECT); print("DEEPGLOBE:", DEEPGLOBE, "|", len(masks), "labelled tiles")
!pip -q install segmentation-models-pytorch
!nvidia-smi -L

PROJECT  : /kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai
DEEPGLOBE: /kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/train | 6226 labelled tiles
GPU 0: Tesla T4 (UUID: GPU-5d415a1c-328c-52a2-6d05-b23bf603e75a)
GPU 1: Tesla T4 (UUID: GPU-388b1519-02a0-bc4b-9f6e-daf0b169a5fd)


In [3]:
!python scripts/prepare_deepglobe.py --src "$DEEPGLOBE" --out /kaggle/working/processed

  val    747 tiles
  test   498 tiles
  train  4981 tiles

Wrote /kaggle/working/processed/manifest.json

Next:
  python -m src.models.train --model unet_r34 --data data/processed --epochs 40
  python -m src.models.train --model segformer_b3 --data data/processed --epochs 40
  python -m src.models.train --model segformer_b3 --data data/processed --no-occlusion


In [4]:
import torch
cands = glob.glob('/kaggle/input/**/best.pt', recursive=True)
chosen = None
for c in cands:
    ck = torch.load(c, map_location="cpu", weights_only=False)
    cfg = ck.get("config") or {}
    print(c, "| arch:", ck.get("args", {}).get("model"), "| epoch:", ck.get("epoch"),
          "| best:", ck.get("best_value"), "| lovasz:", cfg.get("lovasz_weight"))
    if ck.get("args", {}).get("model") == "segformer_b3" and ck.get("epoch") == 21:
        chosen = c
assert chosen, "Fine-tuned SegFormer (epoch 21) not found - attach that notebook's output"
OUT = "/kaggle/working/out/segformer_b3_ft"
os.makedirs(OUT, exist_ok=True)
shutil.copy(chosen, OUT + "/best.pt")
print("USING:", chosen)

/kaggle/input/datasets/kashaksingh/segformer-ft-model/best.pt | arch: segformer_b3 | epoch: 21 | best: 0.6833 | lovasz: 0.5
/kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai/checkpoints/segformer_b3_ft/best.pt | arch: segformer_b3 | epoch: 21 | best: 0.6833 | lovasz: 0.5
/kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai/checkpoints/unet_r34/best.pt | arch: unet_r34 | epoch: 36 | best: 0.65492 | lovasz: None
USING: /kaggle/input/datasets/kashaksingh/rr-code/route-resilience-ai/checkpoints/segformer_b3_ft/best.pt


In [5]:
!python scripts/evaluate_checkpoint.py --checkpoint $OUT/best.pt --data /kaggle/working/processed --out $OUT/report --skip-classical
os.remove(OUT + "/best.pt")          # keep the output small: report only
shutil.make_archive(f"/kaggle/working/{RUN}_report", "zip", OUT)
shutil.rmtree("/kaggle/working/processed", ignore_errors=True)
print("DONE")

  Post-training evaluation
  checkpoint : /kaggle/working/out/segformer_b3_ft/best.pt
  arch       : segformer_b3   epoch 21   best cldice=0.6833
  device     : cuda   threshold 0.5   tile 512
  git        : None

[val] 747 tiles
       IoU 0.6536  Dice 0.78119  P 0.75978  R 0.82762  clDice 0.85819  PixAcc 0.98147

[test] 498 tiles
       IoU 0.66883  Dice 0.79174  P 0.76454  R 0.8443  clDice 0.86745  PixAcc 0.98244

  val - test IoU gap: -0.01523  (consistent)

[occlusion] 498 tiles x 4 conditions
       normal   IoU 0.66883  Dice 0.79174  P 0.76454  R 0.8443  OccRecall None
       canopy   IoU 0.65781  Dice 0.78349  P 0.76087  R 0.83203  OccRecall 0.83267
       partial  IoU 0.70289  Dice 0.81867  P 0.78949  R 0.86825  OccRecall 0.90403
       clutter  IoU 0.69517  Dice 0.81406  P 0.7963  R 0.84939  OccRecall 0.85534

Wrote /kaggle/working/out/segformer_b3_ft/report/evaluation_report.json and .md

# Evaluation report

- checkpoint: `/kaggle/working/out/segformer_b3_ft/best.pt`
- arch